Last updated by Paulo Girardi on september 20, 2025 

paulogirardi@estudante.ufscar.br

# Alzheimer's Disease Neuroimaging Initiative (ADNI)


## Definition

In [1]:
# Packages 

import pandas as pd
import os
from itertools import combinations



In [2]:
# MERGE
# coleções, diagnósticos e dados técnicos 

# Carrega a planilha de coleções, ADNI_mycollections.csv, a planilha de diagnósticos, DXSUM.csv
def load_csv(collections_path, diagnosis_path, idasearch_path, adas_path, cdr_path, mmse_path, faq_path):
    collections_df = pd.read_csv(collections_path)
    diagnosis_df = pd.read_csv(diagnosis_path)
    idasearch_df = pd.read_csv(idasearch_path)
    adas_df = pd.read_csv(adas_path)
    cdr_df = pd.read_csv(cdr_path)
    mmse_df = pd.read_csv(mmse_path)
    faq_df = pd.read_csv(faq_path)

    # print('Planilhas RAW da ADNI: mycollections.csv, idasearch.csv, dxsum.csv\n')
    # print(f"mycollections.csv -> Pacientes da coleção: {collections_df['Subject'].nunique()}")
    # print(f"mycollections.csv -> Imagens da coleção: {len(collections_df)}")
    # print(f"idasearch.csv -> Informações técnicas: {len(idasearch_df)}")
    # print(f"dxsum.csv -> Pacientes na planilha diagnósticos: {diagnosis_df['PTID'].nunique()}")
    # print(f"dxsum.csv -> Quantidade de diagnósticos: {len(diagnosis_df)}")
    # print('-'*50)

    return collections_df, diagnosis_df, idasearch_df, adas_df, cdr_df, mmse_df, faq_df

# Renomear colunas e converter valores dos DataFrames de coleções e diagnósticos para manter apenas as informações úteis e facilitar a manipulação dos dados. 
def rename_collections_header(collections_df):
    collections_df = collections_df.copy()
    collections_df = collections_df.rename(columns={
        'Image Data ID': 'ID_IMG',
        'Subject': 'ID_PT',
        'Sex': 'SEX',
        'Age': 'AGE',
        'Group': 'BASELINE',
        'Acq Date': 'MRI_DATE',
        'Description': 'DESCRIPTION',
        'Visit': 'VISIT'
    })
    # Remover prefixo 'I' para compatibilizar com o datasheet
    collections_df['ID_IMG'] = collections_df['ID_IMG'].astype(str)
    collections_df['MRI_DATE'] = pd.to_datetime(collections_df['MRI_DATE'], format='%m/%d/%Y', errors='coerce')

    return collections_df[['ID_PT', 'ID_IMG', 'BASELINE', 'DESCRIPTION', 'SEX', 'AGE', 'MRI_DATE', 'VISIT']]

# Renomear colunas de diagnósticos e manter apenas as informações principais. 
def rename_diagnosis_header(diagnosis_df): 
    renamed_diagnosis_df = diagnosis_df.rename(columns={
        'PTID': 'ID_PT',
        'EXAMDATE': 'DIAG_DATE',
        'DIAGNOSIS': 'DIAG',
    })
    
    # Converter os valores da coluna 'DIAG'
    renamed_diagnosis_df['DIAG'] = renamed_diagnosis_df['DIAG'].replace({
        1: 'CN',
        2: 'MCI',
        3: 'AD'
    })
    
    renamed_diagnosis_df = renamed_diagnosis_df[['ID_PT', 'DIAG', 'DIAG_DATE', 'DXOTHDEM']]
    
    return renamed_diagnosis_df

def rename_adas_header(adas_df): 

    renamed_adas_df = adas_df.rename(columns={
        'TOTSCORE': 'ADAS_SCORE',
        'PTID': 'ID_PT',
        'VISDATE': 'DIAG_DATE',       
    })

    renamed_adas_df = renamed_adas_df[['ID_PT', 'ADAS_SCORE', 'DIAG_DATE']]

    return renamed_adas_df

def rename_cdr_header(cdr_df):
    
    renamed_cdr_df = cdr_df.rename(columns={
        'PTID': 'ID_PT',
        'VISDATE': 'DIAG_DATE',
        'CDGLOBAL': 'CDR_GLOBAL',
        'CDRSB': 'CDR_SB',       
    })

    renamed_cdr_df = renamed_cdr_df[['ID_PT', 'DIAG_DATE', 'CDR_GLOBAL', 'CDR_SB']]
    
    return renamed_cdr_df

def rename_mmse_header(mmse_df):
    
    renamed_mmse_df = mmse_df.rename(columns={
        'PTID': 'ID_PT',
        'VISDATE': 'DIAG_DATE',
        'MMSCORE': 'MMSE_SCORE',  
    })

    renamed_mmse_df = renamed_mmse_df[['ID_PT', 'DIAG_DATE', 'MMSE_SCORE']]
    
    return renamed_mmse_df

def rename_faq_header(faq_df):
    
    renamed_faq_df = faq_df.rename(columns={
        'PTID': 'ID_PT',
        'VISDATE': 'DIAG_DATE',
        'FAQTOTAL': 'FAQ_SCORE',  
    })

    renamed_faq_df = renamed_faq_df[['ID_PT', 'DIAG_DATE', 'FAQ_SCORE']]
    
    return renamed_faq_df

# Ajustar a coluna 'Imaging Protocol' do DataFrame de dados tecnicos, idasearch_df, criando novas colunas para cada descrição e preenchendo-as com os valores correspondentes.
def adjust_imaging_protocol(idasearch_df):
    # Definir as novas colunas
    new_columns = ['Mfg Model', 'Slice Thickness', 'Matrix Z', 'Field Strength', 'Manufacturer']
    
    # Inicializar as novas colunas com valores vazios
    for col in new_columns:
        idasearch_df[col] = ''
    
    # Iterar sobre as linhas do DataFrame
    for index, row in idasearch_df.iterrows():
        # Obter o valor da coluna 'Imaging Protocol'
        imaging_protocol = row['Imaging Protocol']
        
        # Dividir a string por ';' para obter as descrições e valores
        parts = imaging_protocol.split(';')
        
        # Iterar sobre as partes e preencher as novas colunas
        for part in parts:
            key, value = part.split('=')
            key = key.strip()
            value = value.strip()
            if key in new_columns:
                idasearch_df.at[index, key] = value
    
    # Remover a coluna original 'Imaging Protocol'
    idasearch_df = idasearch_df.drop(columns=['Imaging Protocol'])
    
    return idasearch_df

# Renomear e reordenar as colunas do DataFrame de dados técnicos, idasearch_df, conforme as novas descrições e ordem especificadas, e adicionar a letra 'I' maiúscula antes dos valores da coluna 'ID_IMG'.
def rename_and_reorder_columns(df):
    # Dicionário de mapeamento das colunas
    column_mapping = {
        'Image ID': 'ID_IMG',
        'Subject ID': 'ID_PT',
        'Study Date': 'MRI_DATE',
        'Phase': 'STUDY',
        'Research Group': 'BASELINE',
        'Weight': 'WEIGHT',
        'Mfg Model': 'MFG_MODEL',
        'Slice Thickness': 'SLICE_THICKNESS',
        'Matrix Z': 'MATRIX_Z',
        'Field Strength': 'FIELD_STRENGTH',
        'Manufacturer': 'MANUFACTURER'
    }

    # Adiciona 'I' ao 'Image ID' se necessário
    if 'Image ID' in df.columns:
        df['Image ID'] = df['Image ID'].astype(str)
        df['Image ID'] = df['Image ID'].apply(lambda x: x if x.startswith('I') else 'I' + x)

    # Renomear as colunas
    df = df.rename(columns=column_mapping)

    # Garantir que as colunas-chave estejam como string
    df['ID_IMG'] = df['ID_IMG'].astype(str)
    df['ID_PT'] = df['ID_PT'].astype(str)

    # Definir nova ordem das colunas
    new_order = [
        'ID_IMG', 'ID_PT', 'MRI_DATE', 'STUDY', 'BASELINE', 'WEIGHT',
        'MFG_MODEL', 'SLICE_THICKNESS', 'MATRIX_Z',
        'FIELD_STRENGTH', 'MANUFACTURER'
    ]

    # Adicionar colunas ausentes com valores nulos
    for col in new_order:
        if col not in df.columns:
            df[col] = None

    # Reordenar
    df = df[new_order]

    return df

# Na planilha das coleções, ADNI_mycollections.csv, remove as linhas da coluna Sex com sexo inválido 'X' e mantém apenas 'M' e 'F'.
def remove_empty_sex(collections_df):
    # Filtrar as linhas onde a coluna 'Sex' não é 'X'
    filtered_collections_df = collections_df[collections_df['SEX'] != 'X']
    
    return filtered_collections_df

# Na planilha de diagnósticos, DXSUM.csv, na coluna 'DXOTHDEM', (Diagnosis of Other Dementia not Alzheimer's Disease) remove linhas com diagnóstico válido para outras demências, dado como valor '1' e mantém apenas diagnósticos para a doença de Alzheimer, dado como valor '-4'.
def filter_dementia(diagnosis_df):
    # Filtrar as linhas onde a coluna 'DXOTHDEM' tem o valor '-4' e 'vazio'
    filtered_diagnosis_df = diagnosis_df[diagnosis_df['DXOTHDEM'] != 1]
    
    return filtered_diagnosis_df
    
# Remover datas vazias na coluna DATE_DIAG da planilha de diagnósticos. 
def remove_empty_dates(diagnosis_df):
    no_empty_date_diagnosis_df = diagnosis_df.dropna(subset=['DIAG_DATE'])
    
    return no_empty_date_diagnosis_df

# Remover diagnósticos vazios na coluna 'DIAG' da planilha de diagnósticos.
def remove_empty_diagnoses(diagnosis_df):
    only_diagnosis_df = diagnosis_df.dropna(subset=['DIAG'])
    
    return only_diagnosis_df

# Identificar os pacientes que reverteram de AD para MCI ou CN, segundo a planilha de diagnósticos da ADNI.
def ad_reversion(diagnosis_df):
    ad_pt_reverted_ls = []
    for id_pt, diag in diagnosis_df.groupby('ID_PT'):
        cn_mci_dates = diag[diag['DIAG'].isin(['CN', 'MCI'])]['DIAG_DATE']
        if cn_mci_dates.empty:
            continue
        ad_dates = diag[diag['DIAG'] == 'AD']['DIAG_DATE']
        if ad_dates.empty:
            continue
        first_ad_date = ad_dates.min()
        if any(cn_mci_dates > first_ad_date):
            ad_pt_reverted_ls.append(id_pt)

    ad_pt_not_reverted_df = diagnosis_df[~diagnosis_df['ID_PT'].isin(ad_pt_reverted_ls)]

    # Contar a quantidade de pacientes e diagnósticos MCI que não reverteram
    print(f"Filtro AD não reversores:\n")
    print(f"Pacientes AD sem reversão: {ad_pt_not_reverted_df['ID_PT'].nunique()}")
    print(f"Diagnósticos AD sem reversão: {len(ad_pt_not_reverted_df)}")
    print('-'*50)
    
    return ad_pt_not_reverted_df

# Identificar os pacientes que reverteram de MCI para CN, segundo a planilha de diagnósticos da ADNI.
def mci_reversion(diagnosis_df):
    mci_pt_reverted_ls = []
    for id_pt, diag in diagnosis_df.groupby('ID_PT'):
        cn_dates = diag[diag['DIAG'].isin(['CN'])]['DIAG_DATE']
        if cn_dates.empty:
            continue
        mci_dates = diag[diag['DIAG'] == 'MCI']['DIAG_DATE']
        if mci_dates.empty:
            continue
        first_mci_date = mci_dates.min()
        if any(cn_dates > first_mci_date):
            mci_pt_reverted_ls.append(id_pt)

    mci_not_reverted_df = diagnosis_df[~diagnosis_df['ID_PT'].isin(mci_pt_reverted_ls)]

    # Contar a quantidade de pacientes e diagnósticos MCI que não reverteram
    print(f"Filtro MCI não reversores:\n")
    print(f"Pacientes MCI sem reversão: {mci_not_reverted_df['ID_PT'].nunique()}")
    print(f"Diagnósticos MCI sem reversão: {len(mci_not_reverted_df)}")
    print('-'*50)
    
    return mci_not_reverted_df

def merge_based_on_dates(collections_df, diagnosis_df,
                         adas_df, cdr_df, mmse_df, faq_df,
                         time_diff_mri_diag):

    # Garantir que as colunas de data sejam datetime
    collections_df['MRI_DATE'] = pd.to_datetime(collections_df['MRI_DATE'], errors='coerce')
    diagnosis_df['DIAG_DATE'] = pd.to_datetime(diagnosis_df['DIAG_DATE'], errors='coerce')
    adas_df['DIAG_DATE']      = pd.to_datetime(adas_df['DIAG_DATE'], errors='coerce')
    cdr_df['DIAG_DATE']       = pd.to_datetime(cdr_df['DIAG_DATE'], errors='coerce')
    mmse_df['DIAG_DATE']      = pd.to_datetime(mmse_df['DIAG_DATE'], errors='coerce')
    faq_df['DIAG_DATE']      = pd.to_datetime(faq_df['DIAG_DATE'], errors='coerce')
    
    # Remover linhas com escores vazios nas planilhas de entrada
    adas_df = adas_df.dropna(subset=['ADAS_SCORE'])
    cdr_df  = cdr_df.dropna(subset=['CDR_GLOBAL', 'CDR_SB'], how='any')
    mmse_df = mmse_df.dropna(subset=['MMSE_SCORE'])
    faq_df  = faq_df.dropna(subset=['FAQ_SCORE'])

    # Para coleções e diagnóstico principal, datas vazias são realmente problema
    if collections_df['MRI_DATE'].isnull().any():
        raise ValueError("Existem valores inválidos na coluna 'MRI_DATE' que não puderam ser convertidos para datetime.")
    if diagnosis_df['DIAG_DATE'].isnull().any():
        raise ValueError("Existem valores inválidos na coluna 'DIAG_DATE' que não puderam ser convertidos para datetime.")

    # Remover linhas sem data nas planilhas de scores
    adas_df = adas_df.dropna(subset=['DIAG_DATE'])
    cdr_df  = cdr_df.dropna(subset=['DIAG_DATE'])
    mmse_df = mmse_df.dropna(subset=['DIAG_DATE'])
    faq_df  = faq_df.dropna(subset=['DIAG_DATE'])
    
    # Garantir que as colunas ID_PT sejam do mesmo tipo
    collections_df['ID_PT'] = collections_df['ID_PT'].astype(str)
    diagnosis_df['ID_PT']   = diagnosis_df['ID_PT'].astype(str)
    adas_df['ID_PT']        = adas_df['ID_PT'].astype(str)
    cdr_df['ID_PT']         = cdr_df['ID_PT'].astype(str)
    mmse_df['ID_PT']        = mmse_df['ID_PT'].astype(str)
    faq_df['ID_PT']         = faq_df['ID_PT'].astype(str)

    # Ordenar os DataFrames pelas colunas de data
    collections_df = collections_df.sort_values('MRI_DATE')
    diagnosis_df   = diagnosis_df.sort_values('DIAG_DATE')
    adas_df        = adas_df.sort_values('DIAG_DATE')
    cdr_df         = cdr_df.sort_values('DIAG_DATE')
    mmse_df        = mmse_df.sort_values('DIAG_DATE')
    faq_df         = faq_df.sort_values('DIAG_DATE')

    # 1) Diagnóstico + ADAS
    merge_results_df = pd.merge_asof(
        diagnosis_df,
        adas_df,
        left_on='DIAG_DATE',
        right_on='DIAG_DATE',
        by='ID_PT',
        direction='nearest'
    )

    # 2) + CDR
    merge_results_df = pd.merge_asof(
        merge_results_df,
        cdr_df,
        left_on='DIAG_DATE',
        right_on='DIAG_DATE',
        by='ID_PT',
        direction='nearest'
    )

    # 3) + MMSE
    merge_results_df = pd.merge_asof(
        merge_results_df,
        mmse_df,
        left_on='DIAG_DATE',
        right_on='DIAG_DATE',
        by='ID_PT',
        direction='nearest'
    )
    
    # 4) + FAQ
    merge_results_df = pd.merge_asof(
        merge_results_df,
        faq_df,
        left_on='DIAG_DATE',
        right_on='DIAG_DATE',
        by='ID_PT',
        direction='nearest'
    )

    # 5) Coleções + (diag + ADAS + CDR + MMSE + FAQ)
    final_merge_results_df = pd.merge_asof(
        collections_df.sort_values('MRI_DATE'),
        merge_results_df.sort_values('DIAG_DATE'),
        left_on='MRI_DATE',
        right_on='DIAG_DATE',
        by='ID_PT',
        direction='nearest'
    )

    # Calcular a diferença de datas em dias
    final_merge_results_df['date_diff'] = (
        final_merge_results_df['DIAG_DATE'] - final_merge_results_df['MRI_DATE']
    ).abs()

    # Filtrar as linhas onde a diferença de data está dentro da janela em meses
    final_merge_results_df = final_merge_results_df[
        final_merge_results_df['date_diff'] <= pd.Timedelta(days=time_diff_mri_diag * 30)
    ]

    # Remover a coluna auxiliar 'date_diff'
    final_merge_results_df = final_merge_results_df.drop(columns=['date_diff'])

    # Selecionar colunas finais
    merged_df = final_merge_results_df[[
        'ID_IMG', 'ID_PT', 'SEX', 'AGE', 'DESCRIPTION',
        'MRI_DATE', 'DIAG_DATE', 'VISIT', 'DIAG',
        'ADAS_SCORE', 'CDR_GLOBAL', 'CDR_SB', 'MMSE_SCORE', 'FAQ_SCORE'
    ]]

    # Remover qualquer linha que ainda tenha score em branco
    merged_df = merged_df.dropna(
        subset=['ADAS_SCORE', 'CDR_GLOBAL', 'CDR_SB', 'MMSE_SCORE', 'FAQ_SCORE'],
        how='any'
    )

    return merged_df

# def keep_repeats(df):
#     """
#     Remove imagens originais (sem 'repeat') quando há duplicatas com o mesmo ID_PT e MRI_DATE.
#     Mantém apenas as imagens marcadas como 'repeat' ou 'rpt'.
    
#     Lógica:
#     1. Normaliza a coluna DESCRIPTION (case-insensitive)
#     2. Marca todas as linhas com 'repeat' ou 'rpt' na descrição
#     3. Para cada grupo (ID_PT, MRI_DATE):
#        - Se há múltiplas imagens, REMOVE todas SEM 'repeat' e MANTÉM as com 'repeat'
#        - Se há apenas uma imagem, MANTÉM independentemente
    
#     Parâmetros:
#     - df: DataFrame com os dados das imagens ADNI
    
#     Retorna:
#     - DataFrame filtrado com imagens de repetição mantidas quando há duplicatas
#     """
#     df = df.copy()

#     # Normalizar a coluna DESCRIPTION para facilitar a identificação (case-insensitive)
#     df["description_lower"] = df["DESCRIPTION"].astype(str).str.lower().fillna("")

#     # Marcar linhas com 'repeat' ou 'rpt' na descrição (captura todas as variações)
#     df["is_repeat"] = df["description_lower"].str.contains(r"repeat|rpt", na=False)

#     # Função para processar cada grupo (ID_PT, MRI_DATE)
#     def filter_group(group):
#         if len(group) == 1:
#             # Se há apenas uma imagem, manter como está
#             return group
#         else:
#             # Se há múltiplas imagens, manter apenas as com 'repeat'
#             repeat_images = group[group["is_repeat"]]
#             if len(repeat_images) > 0:
#                 return repeat_images
#             else:
#                 # Se nenhuma tem 'repeat', manter a primeira como fallback
#                 return group.iloc[[0]]

#     # Aplicar o filtro para cada combinação de (ID_PT, MRI_DATE)
#     result_df = df.groupby(["ID_PT", "MRI_DATE"], group_keys=False).apply(filter_group)

#     # Remover colunas auxiliares criadas durante o processo
#     result_df = result_df.drop(columns=["description_lower", "is_repeat"])
    
#     return result_df  

# Arredondar os valores de 2.9 para 3.0 na coluna FIELD_STRENGTH do arquivo adnimerged_df final.
def round_field_strength(df):
    if 'FIELD_STRENGTH' in df.columns:
        # Substitui 2.9 por 3.0 na coluna FIELD_STRENGTH
        df['FIELD_STRENGTH'] = df['FIELD_STRENGTH'].apply(lambda x: 3.0 if float(x) > 2.5 else x)
    else:
        print("A coluna 'FIELD_STRENGTH' não existe no DataFrame.")
    
    return df

# Salvar o DataFrame mesclado em um arquivo CSV chamado "adni_merged.csv".
def save_to_csv(adnimerged_df, output_path):
    adnimerged_df.to_csv(output_path, index=False)
    
def merge_with_idasearch(collections_diagnosis_merged_df, idasearch_renamed_df):
    collections_diagnosis_merged_df = collections_diagnosis_merged_df.copy()
    idasearch_renamed_df = idasearch_renamed_df.copy()

    # Garantir tipos
    for df in (collections_diagnosis_merged_df, idasearch_renamed_df):
        df['ID_IMG'] = df['ID_IMG'].astype(str)
        df['ID_PT']  = df['ID_PT'].astype(str)

    # Merge com sufixos controlados
    merged_df = pd.merge(
        collections_diagnosis_merged_df,
        idasearch_renamed_df,
        on=['ID_IMG', 'ID_PT'],
        how='left',
        suffixes=('_COLL', '_IDA')
    )

    # Escolher qual MRI_DATE manter (recomendado: da coleção)
    if 'MRI_DATE_COLL' in merged_df.columns:
        merged_df['MRI_DATE'] = merged_df['MRI_DATE_COLL']
    elif 'MRI_DATE_IDA' in merged_df.columns:
        merged_df['MRI_DATE'] = merged_df['MRI_DATE_IDA']

    # Remover colunas duplicadas antigas, se existirem
    for c in ['MRI_DATE_COLL', 'MRI_DATE_IDA']:
        if c in merged_df.columns:
            merged_df = merged_df.drop(columns=[c])

    # Reordenação final
    new_order = [
        'STUDY', 'DESCRIPTION', 'ID_IMG', 'ID_PT', 'SEX', 'AGE', 'WEIGHT',
        'MRI_DATE', 'DIAG_DATE', 'VISIT','BASELINE', 'DIAG', 'ADAS_SCORE', 'CDR_GLOBAL', 'CDR_SB', 'MMSE_SCORE', 'FAQ_SCORE',
        'FIELD_STRENGTH', 'SLICE_THICKNESS', 'MATRIX_Z',
        'MANUFACTURER', 'MFG_MODEL'
    ]

    for col in new_order:
        if col not in merged_df.columns:
            merged_df[col] = None

    merged_df = merged_df[new_order]
    return merged_df

def main_merge(collections_path, diagnosis_path, idasearch_path, adas_path, cdr_path, mmse_path, faq_path, output_path, time_diff_mri_diag):
    # Carregar os dados
    collections_df, diagnosis_df, idasearch_df, adas_df, cdr_df, mmse_df, faq_df = load_csv(collections_path, diagnosis_path, idasearch_path, adas_path, cdr_path, mmse_path, faq_path)

    # Renomear colunas do DataFrame de coleções
    collections_df = rename_collections_header(collections_df)
    diagnosis_df = rename_diagnosis_header(diagnosis_df)
    adas_df = rename_adas_header(adas_df)
    cdr_df = rename_cdr_header(cdr_df)
    mmse_df = rename_mmse_header(mmse_df)
    faq_df = rename_faq_header(faq_df)
        
    # Remover entradas com sexo inválido
    collections_df = remove_empty_sex(collections_df)
    
    # Filtrar diagnósticos de demências que não são Alzheimer
    diagnosis_df = filter_dementia(diagnosis_df)

    # Remover diagnósticos em branco
    diagnosis_df = remove_empty_diagnoses(diagnosis_df)

    # Remover diagnósticos sem data
    diagnosis_df = remove_empty_dates(diagnosis_df)

    # Remover pacientes que reverteram os diagnósticos de AD para MCI ou CN
    diagnosis_df = ad_reversion(diagnosis_df)

    # Remover pacientes que reverteram os diagnósticos de MCI para CN
    diagnosis_df = mci_reversion(diagnosis_df)

    # Ajustar o protocolo de imagem no DataFrame idasearch
    idasearch_df = adjust_imaging_protocol(idasearch_df)

    # Renomear e reordenar colunas no DataFrame idasearch
    idasearch_df = rename_and_reorder_columns(idasearch_df)

    # Salvar os DataFrames como arquivos CSV
    os.makedirs(output_path, exist_ok=True)
    save_to_csv(collections_df, output_path + 'collections_df.csv')
    save_to_csv(diagnosis_df, output_path + 'diagnosis_df.csv')
    save_to_csv(idasearch_df, output_path + 'idasearch_df.csv')
    save_to_csv(adas_df, output_path + 'adas_df.csv')
    save_to_csv(cdr_df, output_path + 'cdr_df.csv')
    save_to_csv(mmse_df, output_path + 'mmse_df.csv')
    save_to_csv(faq_df, output_path + 'faq_df.csv')

    collections_diagnosis_merged_df = merge_based_on_dates(
    collections_df, diagnosis_df,  
    adas_df, cdr_df, mmse_df, faq_df,    
    time_diff_mri_diag
)

    # Estatísticas
    id_img_merged = list(collections_diagnosis_merged_df['ID_IMG'].unique())
    initial_id_pt_count = collections_df['ID_PT'].nunique()
    removed_id_pt_count = initial_id_pt_count - collections_diagnosis_merged_df['ID_PT'].nunique()
    print(f"Quantidade de pacientes removidos após o merge: {removed_id_pt_count}")

    initial_id_img_count = collections_df['ID_IMG'].nunique()
    removed_id_img_count = initial_id_img_count - len(id_img_merged)
    print(f"Quantidade de imagens removidas após o merge: {removed_id_img_count}")
    
    # Mesclar com informações técnicas
    adnimerged_df = merge_with_idasearch(collections_diagnosis_merged_df, idasearch_df)

    # Arredondar valores de campo magnético
    adnimerged_filtered_df = round_field_strength(adnimerged_df)
    
    print(f"Quantidade de imagens antes do filtro de repetição: {len(adnimerged_df)}")
    # # Manter apenas imagens de repetição quando houver duplicatas
    # adnimerged_filtered_df = keep_repeats(adnimerged_filtered_df)

    print(f"Quantidade de imagens após o filtro de repetição: {len(adnimerged_filtered_df)}")
    
    # Salvar resultado final
    save_to_csv(adnimerged_filtered_df, output_path + 'adnimerged.csv')
    print(f"Quantidade de imagens rotuladas: {len(adnimerged_filtered_df)}")

    files_to_delete = [
        'collections_df.csv',
        'diagnosis_df.csv',
        'idasearch_df.csv',
        'adas_df.csv',
        'cdr_df.csv',
        'mmse_df.csv',
        'faq_df.csv'
    ]

    for filename in files_to_delete:
        filepath = os.path.join(output_path, filename)
        if os.path.exists(filepath):
            os.remove(filepath)
            print(f"✓ Deletado: {filename}")

    return adnimerged_filtered_df



In [3]:
# CRITERIA
# INCLUSION & EXCLUSION

# Carrega o arquivo CSV adnimerged.csv e ordenar por ID_PT e DIAG_DATE. E retorna um DataFrame chamado adnimerged_df.
def load_data(file_path):
    merged_df = pd.read_csv(file_path)
    merged_df = merged_df.sort_values(by=['ID_PT', 'DIAG_DATE']) # Ordena o DataFrame por ID_PT e DIAG_DATE em ordem crescente
    return merged_df

# Filtra todos os diagnósticos CN e imprime o número de imagens.
def cn_filter(df):
    # Ordena o DataFrame por ID_PT e DIAG_DATE em ordem crescente
    df = df.sort_values(by=['ID_PT', 'DIAG_DATE'])
    cn_df = df[df['DIAG'] == 'CN']
    return cn_df

# Separa o DataFrame em dois grupos: sMCI e pMCI.
def smci_pmci_create(df):
    # Ordena o DataFrame por ID_PT e DIAG_DATE em ordem crescente
    df = df.sort_values(by=['ID_PT', 'DIAG_DATE'])
    smci_patients = []
    pmci_patients = []
    # Itera sobre cada paciente único
    for patient_id in df['ID_PT'].unique():
        patient_data = df[df['ID_PT'] == patient_id]
        diagnoses = patient_data['DIAG'].unique()
        if 'MCI' in diagnoses:
            if 'AD' in diagnoses:
                pmci_patients.append(patient_data[patient_data['DIAG'].isin(['MCI', 'AD'])])
            else:
                smci_patients.append(patient_data[patient_data['DIAG'] == 'MCI'])
    smci_df = pd.concat(smci_patients).drop_duplicates()
    pmci_df = pd.concat(pmci_patients).drop_duplicates()
    return smci_df, pmci_df

# Filtra todos os diagnósticos MCI e apenas o primeiro AD para cada paciente que tenha ambos os diagnósticos.
def first_ad_pmci_only(df):
    filtered_patients = []
    # Ordena o DataFrame por ID_PT e DIAG_DATE
    df = df.sort_values(by=['ID_PT', 'DIAG_DATE'])
    # Itera sobre cada paciente único
    for patient_id in df['ID_PT'].unique():
        patient_data = df[df['ID_PT'] == patient_id]
        mci_data = patient_data[patient_data['DIAG'] == 'MCI']
        ad_data = patient_data[patient_data['DIAG'] == 'AD'].head(1)  # Seleciona apenas o primeiro diagnóstico AD
        if not mci_data.empty and not ad_data.empty:
            filtered_patients.append(mci_data)
            filtered_patients.append(ad_data)
    only_1st_ad_df = pd.concat(filtered_patients).drop_duplicates()
    return only_1st_ad_df

# Filtra os diagnósticos AD no conjunto pMCI com base na diferença de tempo entre o diagnóstico AD e o primeiro diagnóstico MCI. Se a diferença de tempo for menor que 12 meses, mantém o diagnóstico AD; caso contrário, remove apenas a imagem com o diagnóstico AD.
def pmci_time_diff(df, time_diff_ad_mci):
    df['DIAG_DATE'] = pd.to_datetime(df['DIAG_DATE'], errors='coerce')
    filtered_patients = []

    for patient_id in df['ID_PT'].unique():
        patient_data = df[df['ID_PT'] == patient_id]
        ad_data = patient_data[patient_data['DIAG'] == 'AD']
        mci_data = patient_data[patient_data['DIAG'] == 'MCI']

        if not mci_data.empty:
            ad_date = ad_data['DIAG_DATE'].iloc[0]
            last_mci_date = mci_data['DIAG_DATE'].max()
            time_difference = (ad_date - last_mci_date).days / 30.0  # Converte dias para meses

            if time_difference <= time_diff_ad_mci:
                filtered_patients.append(ad_data)

        filtered_patients.append(mci_data)  # Adiciona todas as imagens MCI

    return pd.concat(filtered_patients).drop_duplicates() if filtered_patients else pd.DataFrame()

def keep_repeats(df):
    """
    Remove imagens originais (sem 'repeat') quando há duplicatas com o mesmo ID_PT e MRI_DATE.
    Mantém apenas as imagens marcadas como 'repeat' ou 'rpt'.
    
    Lógica:
    1. Normaliza a coluna DESCRIPTION (case-insensitive)
    2. Marca todas as linhas com 'repeat' ou 'rpt' na descrição
    3. Para cada grupo (ID_PT, MRI_DATE):
       - Se há múltiplas imagens, REMOVE todas SEM 'repeat' e MANTÉM as com 'repeat'
       - Se há apenas uma imagem, MANTÉM independentemente
    
    Parâmetros:
    - df: DataFrame com os dados das imagens ADNI
    
    Retorna:
    - DataFrame filtrado com imagens de repetição mantidas quando há duplicatas
    """
    df = df.copy()

    # Normalizar a coluna DESCRIPTION para facilitar a identificação (case-insensitive)
    df["description_lower"] = df["DESCRIPTION"].astype(str).str.lower().fillna("")

    # Marcar linhas com 'repeat' ou 'rpt' na descrição (captura todas as variações)
    df["is_repeat"] = df["description_lower"].str.contains(r"repeat|rpt", na=False)

    # Função para processar cada grupo (ID_PT, MRI_DATE)
    def filter_group(group):
        if len(group) == 1:
            # Se há apenas uma imagem, manter como está
            return group
        else:
            # Se há múltiplas imagens, manter apenas as com 'repeat'
            repeat_images = group[group["is_repeat"]]
            if len(repeat_images) > 0:
                return repeat_images
            else:
                # Se nenhuma tem 'repeat', manter a primeira como fallback
                return group.iloc[[0]]

    # Aplicar o filtro para cada combinação de (ID_PT, MRI_DATE)
    result_df = df.groupby(["ID_PT", "MRI_DATE"], group_keys=False).apply(filter_group)

    # Remover colunas auxiliares criadas durante o processo
    result_df = result_df.drop(columns=["description_lower", "is_repeat"])
    
    return result_df  

# Define apenas 1 tipo de campo magnético apenas se houver imagens de RM na mesma data pro mesmo paciente.
def no_duplication_filter(df, MFS):
    df = df.copy()
    df['MRI_DATE'] = pd.to_datetime(df['MRI_DATE'], errors='coerce')

    # Agrupa por paciente para tratar cada um separadamente
    filtered_dfs = []

    for pt_id, group in df.groupby('ID_PT'):
        # Identifica se há datas duplicadas para o mesmo paciente
        duplicated_dates = group[group.duplicated(subset=['MRI_DATE'], keep=False)]
        non_duplicated = group[~group['MRI_DATE'].isin(duplicated_dates['MRI_DATE'])]

        # Para datas duplicadas, mantém apenas com o MFS desejado
        duplicated_filtered = duplicated_dates[duplicated_dates['FIELD_STRENGTH'] == MFS]
        duplicated_filtered = duplicated_filtered.drop_duplicates(subset=['MRI_DATE'], keep='first')

        result = pd.concat([non_duplicated, duplicated_filtered])
        filtered_dfs.append(result)

    # Junta todos os pacientes novamente
    return pd.concat(filtered_dfs).reset_index(drop=True)

# janelas deslizantes avançando imagem por imagem, considerando um intervalo de tempo e um número mínimo de imagens.
def forward_sliding_window(df, time_range=36, min_img=3):

    df = df.copy()
    df['DIAG_DATE'] = pd.to_datetime(df['DIAG_DATE'], errors='coerce')
    windowed_data = {}

    for patient_id in df['ID_PT'].unique():
        patient_data = df[df['ID_PT'] == patient_id].sort_values(by='DIAG_DATE')
        for start_date in patient_data['DIAG_DATE']:
            end_date = start_date + pd.DateOffset(months=time_range)
            window = patient_data[(patient_data['DIAG_DATE'] >= start_date) & (patient_data['DIAG_DATE'] < end_date)]
            if len(window) >= min_img:
                key = f"{patient_id}_{start_date.date()}_{end_date.date()}"
                windowed_data[key] = window

    return windowed_data


# janelas voltam no tempo, avançando imagem por imagem por paciente, com separação entre pMCI e sMCI.
# As janelas são criadas de trás para frente, começando pela data do diagnóstico AD
def backward_sliding_window(df, time_range=36, min_img=3):

    df = df.copy()
    df['DIAG_DATE'] = pd.to_datetime(df['DIAG_DATE'], errors='coerce')
    df = df.sort_values(by=['ID_PT', 'DIAG_DATE', 'ID_IMG'], ascending=False)

    windowed_data_pmci = {}
    windowed_data_smci = {}

    for patient_id in df['ID_PT'].unique():
        patient_data = df[df['ID_PT'] == patient_id].sort_values(by='DIAG_DATE', ascending=False)
        for end_date in patient_data['DIAG_DATE']:
            start_date = end_date - pd.DateOffset(months=time_range)
            window = patient_data[(patient_data['DIAG_DATE'] <= end_date) & (patient_data['DIAG_DATE'] > start_date)]
            if len(window) >= min_img:
                key = f"{patient_id}_{end_date.date()}_{start_date.date()}"
                if 'AD' in window['DIAG'].values:
                    windowed_data_pmci[key] = window
                else:
                    windowed_data_smci[key] = window

    return windowed_data_pmci, windowed_data_smci

# remover imagens redundantes oriundas das janelas deslizantes, mantendo apenas as imagens que não foram usadas em janelas anteriores.
def remove_redundant_images_from_windows(window_dict, min_img=3):
    seen_ids = set()
    filtered_dict = {}

    for key, df in window_dict.items():
        ids = set(df['ID_IMG'])
        unique_ids = ids - seen_ids
        filtered_df = df[df['ID_IMG'].isin(unique_ids)]

        if len(filtered_df) >= min_img:
            filtered_dict[key] = filtered_df
            seen_ids.update(unique_ids)

    return filtered_dict

# Combina dois dicionários de dados sMCI em um único dicionário.
def combine_smci_dicts(smci_dict, smci_from_pmci_dict):
    smci_all_dict = {**smci_dict, **smci_from_pmci_dict}

    return smci_all_dict

# Limpa duplicatas exatas do DataFrame e resolve conflitos entre sMCI e pMCI, priorizando pMCI.
def clean_dicts_prioritize_pmci(cn_dict, smci_dict, pmci_dict):

    # Adicionar coluna de grupo a cada DataFrame
    cn_dfs = [df.assign(GROUP='CN') for df in cn_dict.values()]
    smci_dfs = [df.assign(GROUP='sMCI') for df in smci_dict.values()]
    pmci_dfs = [df.assign(GROUP='pMCI') for df in pmci_dict.values()]

    # Concatenar tudo em um único DataFrame
    df = pd.concat(cn_dfs + smci_dfs + pmci_dfs, ignore_index=True)

    # Remover duplicatas exatas
    df = df.drop_duplicates()

    # Resolver conflitos entre pMCI e sMCI
    def resolve_conflict(subdf):
        if 'pMCI' in subdf['GROUP'].values:
            return subdf[subdf['GROUP'] != 'sMCI']
        return subdf

    df_resolved = df.groupby('ID_IMG', group_keys=False).apply(resolve_conflict)

    # Separar novamente em dicionários
    cn_dict_cleaned = {}
    smci_dict_cleaned = {}
    pmci_dict_cleaned = {}

    for i, (key, group_df) in enumerate(df_resolved.groupby(['ID_PT', 'GROUP'])):
        id_pt, group = key
        entry_key = f"{id_pt}_{i}"
        if group == 'CN':
            cn_dict_cleaned[entry_key] = group_df.drop(columns=['GROUP'])
        elif group == 'sMCI':
            smci_dict_cleaned[entry_key] = group_df.drop(columns=['GROUP'])
        elif group == 'pMCI':
            pmci_dict_cleaned[entry_key] = group_df.drop(columns=['GROUP'])

    return cn_dict_cleaned, smci_dict_cleaned, pmci_dict_cleaned

# Salva os dados das imagens em um arquivo de texto, onde cada linha representa uma imagem individual com informações detalhadas, incluindo ID do paciente, ID da imagem, diagnóstico, sexo, idade, data de aquisição e grupo (CN, sMCI, pMCI).
def save_image_data_to_txt(output_path, cn_dict, smci_dict, pmci_dict):
    data = []

    def process_group(dct, group_name):
        for key, df in dct.items():
            df = df.copy()
            df['MRI_DATE'] = pd.to_datetime(df['MRI_DATE'], errors='coerce')

            # Verifica se existem ao menos 3 imagens válidas para formar uma trinca
            if df['ID_IMG'].nunique() < 3:
                continue  # Ignora esse paciente/conjunto

            for _, row in df.iterrows():
                data.append((
                    row['ID_PT'],
                    row['ID_IMG'],
                    row['DIAG'],
                    row['SEX'],
                    row['AGE'],
                    row['MRI_DATE'],
                    group_name
                ))

    process_group(cn_dict, 'CN')
    process_group(smci_dict, 'sMCI')
    process_group(pmci_dict, 'pMCI')

    # Ordena por ID_PT e MRI_DATE
    data.sort(key=lambda x: (x[0], x[5]))

    with open(output_path, 'w') as file:
        file.write("ID_PT,ID_IMG,DIAG,SEX,AGE,MRI_DATE,GROUP\n")
        for id_pt, id_img, diag, sex, age, mri_date, group in data:
            mri_date_str = mri_date.strftime('%Y-%m-%d') if pd.notnull(mri_date) else 'NA'
            file.write(f"{id_pt},{id_img},{diag},{sex},{age},{mri_date_str},{group}\n")

# Salva as trincas de imagem em um arquivo .txt contendo todas as informações dos metadados de cada imagem em cada trinca, incluindo o número da combinação e o grupo (CN, sMCI, pMCI).
def save_cj_to_txt(path, cn_dict, smci_dict, pmci_dict):
    """
    Salva as trincas de imagem em um arquivo .txt contendo apenas os metadados
    relevantes para cada imagem em cada trinca.
    """
    import pandas as pd

    all_data = []

    def process_dict(comb_dict, group_label):
        for patient_key, list_of_triples in comb_dict.items():
            for i, triple_df in enumerate(list_of_triples, start=1):
                triple_df = triple_df.copy()
                triple_df['COMBINATION_NUMBER'] = i
                triple_df['GROUP'] = group_label

                # Selecionar apenas as colunas relevantes
                cols = ['ID_PT', 'ID_IMG', 'DIAG', 'SEX', 'AGE', 'MRI_DATE',
                        'GROUP', 'COMBINATION_NUMBER']
                cols_existentes = [c for c in cols if c in triple_df.columns]
                all_data.append(triple_df[cols_existentes])

    process_dict(cn_dict, 'CN')
    process_dict(smci_dict, 'sMCI')
    process_dict(pmci_dict, 'pMCI')

    final_df = pd.concat(all_data, ignore_index=True)
    final_df.to_csv(path, index=False)

# Filtra os dados por sexo.
def filter_by_sex(data, sex):
    if sex == 'both':
        return data
    filtered_data = {k: v for k, v in data.items() if v.iloc[0]['SEX'] == sex}
    return filtered_data

# Seletor de abordagem: 1=combinatória, 2=sequências consecutivas, 3=primeira janela+combinatória, 4=primeira janela+consecutivas
def get_generate_image_combinations(abordagem):
    """Retorna a função de geração de combinações para a abordagem escolhida (1, 2, 3 ou 4)."""
    return {
        1: generate_image_combinations_abordagem_1,
        2: generate_image_combinations_abordagem_2,
        3: generate_image_combinations_abordagem_3,
        4: generate_image_combinations_abordagem_4,
    }.get(abordagem)

# Abordagem 1: Para um conjunto [1,2,3,4] gera [1,2,3], [1,2,4], [1,3,4], [2,3,4].
def generate_image_combinations_abordagem_1(data_dict, num_images=3):
    """
    Gera combinações sequenciais de imagens ao longo do tempo para cada paciente.
    Para um conjunto [1,2,3,4] gera [1,2,3], [1,2,4], [1,3,4], [2,3,4].

    Parâmetros:
    - data_dict: Dicionário com dados por paciente
    - num_images: Número de imagens por conjunto (padrão: 3)

    Retorna:
    - new_data_dict: Dicionário com conjuntos sequenciais por paciente
    """

    new_data_dict = {}

    for patient_id, df in data_dict.items():
        if len(df) >= num_images:
            # Ordenar por data para manter consistência temporal
            df_sorted = df.sort_values(by='MRI_DATE').reset_index(drop=True)

            # Gera combinações baseadas no número de imagens
            combs = list(combinations(range(len(df_sorted)), num_images))

            # Armazena as combinações como DataFrames
            comb_dfs = [df_sorted.iloc[list(idx)].reset_index(drop=True) for idx in combs]
            new_data_dict[patient_id] = comb_dfs

    return new_data_dict

# Abordagem 2: Para um conjunto [1,2,3,4,5] gera [1,2,3], [2,3,4], [3,4,5] (N-2 conjuntos).
def generate_image_combinations_abordagem_2(data_dict, num_images=3):
    """
    Gera combinações sequenciais de imagens ao longo do tempo para cada paciente.
    Para um conjunto [1,2,3,4,5] gera [1,2,3], [2,3,4], [3,4,5] (N-2 conjuntos).

    Parâmetros:
    - data_dict: Dicionário com dados por paciente
    - num_images: Número de imagens por conjunto (padrão: 3)

    Retorna:
    - new_data_dict: Dicionário com conjuntos sequenciais por paciente
    """

    new_data_dict = {}

    for patient_id, df in data_dict.items():
        if len(df) >= num_images:
            # Ordenar por data para manter consistência temporal
            df_sorted = df.sort_values(by='MRI_DATE').reset_index(drop=True)

            # Gerar sequências temporais consecutivas
            sequential_combinations = []

            # Para N imagens, gerar N-2 conjuntos sequenciais
            for i in range(len(df_sorted) - num_images + 1):
                # Selecionar num_images consecutivas começando do índice i
                sequential_df = df_sorted.iloc[i:i + num_images].reset_index(drop=True)
                sequential_combinations.append(sequential_df)

            # Armazenar apenas se houver pelo menos um conjunto válido
            if sequential_combinations:
                new_data_dict[patient_id] = sequential_combinations

    return new_data_dict

# Abordagem 3: Primeira janela + Combinatória (Abordagem 1)
def generate_image_combinations_abordagem_3(data_dict, num_images=3):
    """
    Abordagem 3A: Primeira janela válida + Combinatória
    - Para sMCI: primeira janela avançando no tempo
    - Para pMCI: primeira janela voltando no tempo
    - Se a janela tem >3 imagens, aplica combinatória (abordagem 1): C(n,3)

    Parâmetros:
    - data_dict: Dicionário com dados por paciente
    - num_images: Número de imagens por conjunto (padrão: 3)

    Retorna:
    - new_data_dict: Dicionário com conjuntos da primeira janela por paciente
    """

    new_data_dict = {}

    for patient_id, df in data_dict.items():
        if len(df) >= num_images:
            # Ordenar por data para manter consistência temporal
            df_sorted = df.sort_values(by='MRI_DATE').reset_index(drop=True)

            # Determinar direção baseada no diagnóstico
            # Se há AD no dataset, é pMCI (volta no tempo)
            # Se não há AD, é sMCI (avança no tempo)
            has_ad = 'AD' in df_sorted['DIAG'].values

            if has_ad:
                # pMCI: volta no tempo - ordenar por data decrescente
                df_sorted = df_sorted.sort_values(by='MRI_DATE', ascending=False).reset_index(drop=True)

            # Encontrar a primeira janela de 36 meses com pelo menos 3 imagens
            first_valid_window = None

            for start_idx in range(len(df_sorted)):
                start_date = df_sorted.iloc[start_idx]['MRI_DATE']

                if has_ad:
                    # pMCI: janela vai do start_date para trás (36 meses antes)
                    end_date = start_date - pd.DateOffset(months=36)
                    window_mask = (df_sorted['MRI_DATE'] <= start_date) & (df_sorted['MRI_DATE'] > end_date)
                else:
                    # sMCI: janela vai do start_date para frente (36 meses depois)
                    end_date = start_date + pd.DateOffset(months=36)
                    window_mask = (df_sorted['MRI_DATE'] >= start_date) & (df_sorted['MRI_DATE'] < end_date)

                window_data = df_sorted[window_mask].copy()

                if len(window_data) >= num_images:
                    # Encontrou a primeira janela válida
                    # Ordenar por data crescente para manter ordem temporal nos conjuntos
                    if has_ad:
                        # Para pMCI, reverter a ordem para ficar crescente
                        window_data = window_data.sort_values(by='MRI_DATE').reset_index(drop=True)

                    first_valid_window = window_data
                    break

            if first_valid_window is not None:
                # APLICAR COMBINATÓRIA (ABORDAGEM 1)
                if len(first_valid_window) == num_images:
                    # Exatamente 3 imagens: usar como está
                    new_data_dict[patient_id] = [first_valid_window]
                else:
                    # Mais de 3 imagens: gerar todas as combinações possíveis C(n,3)
                    combs = list(combinations(first_valid_window.index, num_images))
                    combination_dfs = [first_valid_window.loc[list(idx)].reset_index(drop=True) for idx in combs]
                    new_data_dict[patient_id] = combination_dfs

    return new_data_dict


# Abordagem 4: Primeira janela + Sequências consecutivas (Abordagem 2)
def generate_image_combinations_abordagem_4(data_dict, num_images=3):
    """
    Abordagem 4: Primeira janela válida + Sequências consecutivas
    - Para sMCI: primeira janela avançando no tempo
    - Para pMCI: primeira janela voltando no tempo
    - Se a janela tem >3 imagens, aplica sequências consecutivas (abordagem 2): N-2 conjuntos

    Parâmetros:
    - data_dict: Dicionário com dados por paciente
    - num_images: Número de imagens por conjunto (padrão: 3)

    Retorna:
    - new_data_dict: Dicionário com conjuntos da primeira janela por paciente
    """

    new_data_dict = {}

    for patient_id, df in data_dict.items():
        if len(df) >= num_images:
            # Ordenar por data para manter consistência temporal
            df_sorted = df.sort_values(by='MRI_DATE').reset_index(drop=True)

            # Determinar direção baseada no diagnóstico
            # Se há AD no dataset, é pMCI (volta no tempo)
            # Se não há AD, é sMCI (avança no tempo)
            has_ad = 'AD' in df_sorted['DIAG'].values

            if has_ad:
                # pMCI: volta no tempo - ordenar por data decrescente
                df_sorted = df_sorted.sort_values(by='MRI_DATE', ascending=False).reset_index(drop=True)

            # Encontrar a primeira janela de 36 meses com pelo menos 3 imagens
            first_valid_window = None

            for start_idx in range(len(df_sorted)):
                start_date = df_sorted.iloc[start_idx]['MRI_DATE']

                if has_ad:
                    # pMCI: janela vai do start_date para trás (36 meses antes)
                    end_date = start_date - pd.DateOffset(months=36)
                    window_mask = (df_sorted['MRI_DATE'] <= start_date) & (df_sorted['MRI_DATE'] > end_date)
                else:
                    # sMCI: janela vai do start_date para frente (36 meses depois)
                    end_date = start_date + pd.DateOffset(months=36)
                    window_mask = (df_sorted['MRI_DATE'] >= start_date) & (df_sorted['MRI_DATE'] < end_date)

                window_data = df_sorted[window_mask].copy()

                if len(window_data) >= num_images:
                    # Encontrou a primeira janela válida
                    # Ordenar por data crescente para manter ordem temporal nos conjuntos
                    if has_ad:
                        # Para pMCI, reverter a ordem para ficar crescente
                        window_data = window_data.sort_values(by='MRI_DATE').reset_index(drop=True)

                    first_valid_window = window_data
                    break

            if first_valid_window is not None:
                # APLICAR SEQUÊNCIAS CONSECUTIVAS (ABORDAGEM 2)
                if len(first_valid_window) == num_images:
                    # Exatamente 3 imagens: usar como está
                    new_data_dict[patient_id] = [first_valid_window]
                else:
                    # Mais de 3 imagens: gerar sequências consecutivas N-2
                    sequential_combinations = []
                    # Para N imagens, gerar N-2 conjuntos sequenciais
                    for i in range(len(first_valid_window) - num_images + 1):
                        # Selecionar num_images consecutivas começando do índice i
                        sequential_df = first_valid_window.iloc[i:i + num_images].reset_index(drop=True)
                        sequential_combinations.append(sequential_df)

                    new_data_dict[patient_id] = sequential_combinations

    return new_data_dict

# Função principal responsavel por chamar todas as outras funções e imprimir os resultados.
def main_criteria(file_path, time_diff_ad_mci=12, mfs=1.5, time_range=36, min_img=3, num_img=3, sex='both', output_path='./', abordagem=1):
    # Carregar os dados
    adnimerged_df = load_data(file_path)
    print('Planilha adnimerged.csv:')
    print(f"Imagens rotuladas: {len(adnimerged_df)}")
    print('-'*50)
    
    adnimerged_df = keep_repeats(adnimerged_df)
    
    adnimerged_df = no_duplication_filter(adnimerged_df, mfs)
    
    adnimerged_df.to_csv(
    "output/adni/adnimerged_filtered.csv",
    index=False,
    encoding="utf-8",  # opcional
)
    
    # Converter a coluna AGE para numérico
    adnimerged_df['AGE'] = pd.to_numeric(adnimerged_df['AGE'], errors='coerce')

    # Após carregar o arquivo adnimerged.csv
    # adnimerged_df = keep_repeats(adnimerged_df)

    # Filtrar pacientes com diagnóstico CN (Cognitivamente Normais)
    cn_df = cn_filter(adnimerged_df)
    print(f"Filtro de diagnósticos CN:\n")
    print(f"Quantidade de pacientes com diagnóstico CN: {len(cn_df['ID_PT'].unique())}")
    print(f"Quantidade de imagens com diagnóstico CN: {len(cn_df)}")
    print('-'*50)

    # Criar subconjuntos de pacientes sMCI (MCI estável) e pMCI (MCI progressivo)
    smci_df, pmci_df = smci_pmci_create(adnimerged_df)
    print(f"Filtro de diagnósticos sMCI e pMCI:\n")
    print('Subconjuntos MCI = sMCI + pMCI:')
    print(f"Pacientes {len(smci_df['ID_PT'].unique())} sMCI + {len(pmci_df['ID_PT'].unique())} pMCI: {len(smci_df['ID_PT'].unique()) + len(pmci_df['ID_PT'].unique())}")
    print(f"Imagens {len(smci_df['ID_IMG'].unique())} sMCI + {len(pmci_df['ID_IMG'].unique())} pMCI: {len(smci_df['ID_IMG']) + len(pmci_df['ID_IMG'])}")
    print('-'*50)

    # Filtrar apenas a primeira imagem AD para pacientes pMCI
    pmci_1st_ad_df = first_ad_pmci_only(pmci_df)
    print(f"Filtro do primeiro diagnóstico AD para pacientes pMCI:\n")
    print(f"Imagens pacientes pmci (todos AD): {len(pmci_df['ID_IMG'])}")
    print(f"Imagens pacientes pmci (apenas 1º AD): {len(pmci_1st_ad_df['ID_IMG'].unique())}")
    print(f"Imagens AD removidas: {len(pmci_df['ID_IMG']) - len(pmci_1st_ad_df['ID_IMG'])}")
    print('-'*50)

    # Filtrar pacientes pMCI com base no tempo entre o último MCI e o primeiro AD
    pmci_after_time_diff_df = pmci_time_diff(pmci_1st_ad_df, time_diff_ad_mci)
    print(f"Filtro de pacientes pMCI com base no tempo entre o último MCI e o primeiro AD:\n")
    print(f"Quantidade de pacientes pMCI após filtro de tempo: {len(pmci_after_time_diff_df['ID_PT'].unique())}")
    print(f"Quantidade de imagens pMCI após filtro de tempo: {len(pmci_after_time_diff_df)}")
    print('-'*50)

    # Filtrar duplicatas e manter apenas as imagens com a intensidade do campo magnético especificada (MFS)

    print("Antes do filtro MFS (no_duplication_filter):\n")
    print(f"CN   — pacientes: {cn_df['ID_PT'].nunique()}, imagens: {len(cn_df)}")
    print(f"sMCI — pacientes: {smci_df['ID_PT'].nunique()}, imagens: {len(smci_df)}")
    print(f"pMCI — pacientes: {pmci_after_time_diff_df['ID_PT'].nunique()}, imagens: {len(pmci_after_time_diff_df)}")
    print('-' * 50)

    # cn_filtered_df = no_duplication_filter(cn_df, mfs)
    # smci_filtered_df = no_duplication_filter(smci_df, mfs)
    # pmci_filtered_df = no_duplication_filter(pmci_after_time_diff_df, mfs)

    # print("Depois do filtro MFS (no_duplication_filter):\n")
    # print(f"CN   — pacientes: {cn_filtered_df['ID_PT'].nunique()}, imagens: {len(cn_filtered_df)}")
    # print(f"sMCI — pacientes: {smci_filtered_df['ID_PT'].nunique()}, imagens: {len(smci_filtered_df)}")
    # print(f"pMCI — pacientes: {pmci_filtered_df['ID_PT'].nunique()}, imagens: {len(pmci_filtered_df)}")
    # print('-' * 50)
    
    # print(f"CN   — imagens removidas: {len(cn_df) - len(cn_filtered_df)}")
    # print(f"sMCI — imagens removidas: {len(smci_df) - len(smci_filtered_df)}")
    # print(f"pMCI — imagens removidas: {len(pmci_after_time_diff_df) - len(pmci_filtered_df)}")

    # Aplicar janela deslizante para pacientes CN
    cn_window_dict = forward_sliding_window(cn_df, time_range, min_img)
    print(f"Aplicação de janela deslizante para pacientes CN:\n")
    print(f"Quantidade de conjuntos CN após janela deslizante: {len(cn_window_dict)}")
    print('-'*50)

    # Aplicar janela deslizante para pacientes sMCI
    smci_window_dict = forward_sliding_window(smci_df, time_range, min_img)
    print(f"Aplicação de janela deslizante para pacientes sMCI:\n")
    print(f"Quantidade de conjuntos sMCI após janela deslizante: {len(smci_window_dict)}")
    print('-'*50)

    # Aplicar janela deslizante para pacientes pMCI
    pmci_window_dict, smci_from_pmci_window_dict = backward_sliding_window(pmci_after_time_diff_df, time_range, min_img)
    print(f"Aplicação de janela deslizante para pacientes pMCI:\n")
    print(f"Quantidade de conjuntos pMCI após janela deslizante: {len(pmci_window_dict)}")
    print(f"Quantidade de conjuntos sMCI derivados de pMCI após janela deslizante: {len(smci_from_pmci_window_dict)}")
    print('-'*50)

    # Combinar os dicionários de janelas deslizantes para sMCI
    smci_all_window_dict = combine_smci_dicts(smci_window_dict, smci_from_pmci_window_dict)
    print(f"Combinação de dicionários de janelas deslizantes para sMCI:\n")
    print(f"Quantidade total de conjuntos sMCI após combinação: {len(smci_all_window_dict)}")
    print('-'*50)

    # Remover imagens redundantes das janelas
    cn_window_dict = remove_redundant_images_from_windows(cn_window_dict,min_img)
    smci_all_window_dict = remove_redundant_images_from_windows(smci_all_window_dict,min_img)
    pmci_window_dict = remove_redundant_images_from_windows(pmci_window_dict,min_img)

    # Filtrar por sexo
    cn_data_dict = filter_by_sex(cn_window_dict, sex)
    smci_all_data_dict = filter_by_sex(smci_all_window_dict, sex)
    pmci_data_dict = filter_by_sex(pmci_window_dict, sex)

    cn_data_dict, smci_all_data_dict, pmci_data_dict = clean_dicts_prioritize_pmci(
    cn_data_dict, smci_all_data_dict, pmci_data_dict)

    # Garante que o diretório de saída exista
    os.makedirs(output_path, exist_ok=True)

    # Validar abordagem (1, 2, 3 ou 4)
    generate_image_combinations = get_generate_image_combinations(abordagem)
    if generate_image_combinations is None:
        raise ValueError("abordagem deve ser 1, 2, 3 ou 4")

    # Caminhos completos de saída (nome do arquivo depende da abordagem)
    image_data_file = os.path.join(output_path, 'image_data.txt')
    cj_data_file = os.path.join(output_path, f'cj_data_abordagem_{abordagem}.txt')

    save_image_data_to_txt(image_data_file, cn_data_dict, smci_all_data_dict, pmci_data_dict)
    print(f"Dados de imagem salvos em {image_data_file}")

    # Gerar combinações de imagens com a abordagem escolhida
    cn_combinations_dict = generate_image_combinations(cn_data_dict, num_img)
    smci_combinations_dict = generate_image_combinations(smci_all_data_dict, num_img)
    pmci_combinations_dict = generate_image_combinations(pmci_data_dict, num_img)

    save_cj_to_txt(cj_data_file, cn_combinations_dict, smci_combinations_dict, pmci_combinations_dict)
    print(f"Conjuntos salvos em {cj_data_file} (abordagem {abordagem})")
    
    cj_path = cj_data_file

    # Carrega cj_data
    cj = pd.read_csv(cj_path)

    # Garante que MRI_DATE está em datetime
    cj['MRI_DATE'] = pd.to_datetime(cj['MRI_DATE'], errors='coerce')

    # Inicializa TIME_PROG com 0 (inteiro)
    cj['TIME_PROG'] = 0

    # Calcula TIME_PROG só para pMCI
    mask_pmci = cj['GROUP'] == 'pMCI'

    # Agrupa por paciente + combinação
    grupo = cj[mask_pmci].groupby(['ID_PT', 'COMBINATION_NUMBER'])

    # Diferença entre última e primeira MRI_DATE em meses (~30.4375 dias/mês)
    time_prog_series = grupo['MRI_DATE'].transform(
        lambda s: (s.max() - s.min()).days / 30.4375
    )

    # Arredonda para inteiro (meses) e mantém suporte a NaN se aparecer
    time_prog_series = time_prog_series.round().astype('Int64')

    # Atribui de volta para cada linha correspondente (apenas pMCI)
    cj.loc[mask_pmci, 'TIME_PROG'] = time_prog_series.values

    # Salva sobrescrevendo o arquivo da abordagem
    cj.to_csv(cj_path, index=False)

    print(f'✅ TIME_PROG calculado (meses inteiros) e salvo em {cj_path}')

    return cn_combinations_dict, smci_combinations_dict, pmci_combinations_dict

## Executar filtros padrões para ADNIMERGED

In [4]:
# RUN MERGE

print('''
##################################################
# MERGE: coleções, diagnósticos e dados técnicos #
##################################################
''')

#Definição das variaveis
# Intervalo de tempo em meses entre data do exame de imagem e a data do diagnóstico
time_diff_mri_diag = 3 
collections_path = 'input/adni/collection.csv'
diagnosis_path = 'input/adni/diagnostics.csv'
idasearch_path = 'input/adni/datasheet.csv'
adas_path = 'input/adni/adas.csv'
cdr_path = 'input/adni/cdr.csv'
mmse_path = 'input/adni/mmse.csv'
faq_path = 'input/adni/faq.csv'
output_path = 'output/adni/'

# # Chamada da função main_merge
adnimerged_filtered_df = main_merge(collections_path, diagnosis_path, idasearch_path, adas_path, cdr_path, mmse_path, faq_path, output_path, time_diff_mri_diag)




##################################################
# MERGE: coleções, diagnósticos e dados técnicos #
##################################################

Filtro AD não reversores:

Pacientes AD sem reversão: 3315
Diagnósticos AD sem reversão: 14215
--------------------------------------------------
Filtro MCI não reversores:

Pacientes MCI sem reversão: 3187
Diagnósticos MCI sem reversão: 13163
--------------------------------------------------
Quantidade de pacientes removidos após o merge: 190
Quantidade de imagens removidas após o merge: 2083
Quantidade de imagens antes do filtro de repetição: 12931
Quantidade de imagens após o filtro de repetição: 12931
Quantidade de imagens rotuladas: 12931
✓ Deletado: collections_df.csv
✓ Deletado: diagnosis_df.csv
✓ Deletado: idasearch_df.csv
✓ Deletado: adas_df.csv
✓ Deletado: cdr_df.csv
✓ Deletado: mmse_df.csv
✓ Deletado: faq_df.csv


## Remove outliers

* Arquivo `datasets/input/adni/outliers_adni.txt` gerado pelo notebook `metricas/detect_outliers.ipynb`



In [5]:
import pandas as pd
import os

def remover_outliers_adni(csv_path, txt_path, output_path):
    """
    Remove outliers do arquivo ADNI merged baseado na lista de IDs no arquivo TXT
    
    Parâmetros:
    - csv_path: Caminho para o arquivo adnimerged.csv
    - txt_path: Caminho para o arquivo outliers_adni.txt
    - output_path: Caminho para salvar o arquivo filtrado
    """
    
    print("="*60)
    print("FILTRO DE OUTLIERS - ADNI")
    print("="*60)
    
    # Carregar o arquivo CSV
    try:
        df_adni = pd.read_csv(csv_path)
        print(f"✅ CSV carregado: {len(df_adni)} registros")
        print(f"   Arquivo: {csv_path}")
    except FileNotFoundError:
        print(f"❌ Erro: Arquivo CSV não encontrado: {csv_path}")
        return None
    except Exception as e:
        print(f"❌ Erro ao carregar CSV: {e}")
        return None
    
    # Carregar o arquivo TXT com outliers
    try:
        with open(txt_path, 'r') as file:
            outliers_ids = []
            for line in file:
                line = line.strip()
                if line:  # Ignorar linhas vazias
                    # Extrair apenas a parte "I+digitos" removendo extensões
                    # Para "I135611.nii.gz" -> "I135611"
                    if line.startswith('I') and '.' in line:
                        # Remove tudo após o primeiro ponto
                        clean_id = line.split('.')[0]
                        outliers_ids.append(clean_id)
                    elif line.startswith('I'):
                        # Caso não tenha extensão, usar como está
                        outliers_ids.append(line)
        
        print(f"✅ TXT carregado: {len(outliers_ids)} IDs de outliers")
        print(f"   Arquivo: {txt_path}")
        
        # Mostrar alguns exemplos dos IDs processados
        if outliers_ids:
            print(f"   Exemplos processados: {outliers_ids[:5]}")
            
    except FileNotFoundError:
        print(f"❌ Erro: Arquivo TXT não encontrado: {txt_path}")
        return None
    except Exception as e:
        print(f"❌ Erro ao carregar TXT: {e}")
        return None
    
    # Verificar se a coluna ID_IMG existe
    if 'ID_IMG' not in df_adni.columns:
        print(f"❌ Erro: Coluna 'ID_IMG' não encontrada no CSV")
        print(f"   Colunas disponíveis: {list(df_adni.columns)}")
        return None
    
    # Converter ID_IMG para string e limpar (remover extensões se existirem)
    df_adni['ID_IMG'] = df_adni['ID_IMG'].astype(str)
    
    # Criar uma coluna auxiliar com IDs limpos para comparação
    df_adni['ID_IMG_CLEAN'] = df_adni['ID_IMG'].apply(lambda x: x.split('.')[0] if '.' in x else x)
    
    # Identificar outliers presentes no DataFrame
    outliers_encontrados = df_adni['ID_IMG_CLEAN'].isin(outliers_ids)
    num_outliers_encontrados = outliers_encontrados.sum()
    
    print(f"\n📊 ANÁLISE DE OUTLIERS:")
    print(f"   • IDs de outliers no TXT: {len(outliers_ids)}")
    print(f"   • Outliers encontrados no CSV: {num_outliers_encontrados}")
    
    if num_outliers_encontrados > 0:
        # Mostrar alguns exemplos dos outliers encontrados
        outliers_df = df_adni[outliers_encontrados]
        print(f"   • Exemplos encontrados:")
        for i, (idx, row) in enumerate(outliers_df.head(3).iterrows()):
            print(f"     - {row['ID_IMG']} -> {row['ID_IMG_CLEAN']} (Paciente: {row.get('ID_PT', 'N/A')})")
        
        # Remover outliers
        df_filtrado = df_adni[~outliers_encontrados].copy()
        registros_removidos = len(df_adni) - len(df_filtrado)
        
        print(f"\n🔄 FILTRO APLICADO:")
        print(f"   • Registros originais: {len(df_adni)}")
        print(f"   • Registros removidos: {registros_removidos}")
        print(f"   • Registros restantes: {len(df_filtrado)}")
        print(f"   • Redução: {(registros_removidos/len(df_adni)*100):.2f}%")
        
    else:
        print(f"   ⚠️ Nenhum outlier foi encontrado no CSV")
        df_filtrado = df_adni.copy()
    
    # Verificar se alguns outliers do TXT não foram encontrados
    outliers_nao_encontrados = []
    for outlier_id in outliers_ids:
        if outlier_id not in df_adni['ID_IMG_CLEAN'].values:
            outliers_nao_encontrados.append(outlier_id)
    
    if outliers_nao_encontrados:
        print(f"\n⚠️ OUTLIERS NÃO ENCONTRADOS NO CSV: {len(outliers_nao_encontrados)}")
        print(f"   Exemplos: {outliers_nao_encontrados[:5]}")
    
    # Remover a coluna auxiliar antes de salvar
    df_filtrado = df_filtrado.drop(columns=['ID_IMG_CLEAN'])
    
    # Salvar arquivo filtrado
    try:
        # Criar diretório se não existir
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        
        # Salvar CSV filtrado
        df_filtrado.to_csv(output_path, index=False)
        print(f"\n✅ Arquivo filtrado salvo: {output_path}")
        print(f"   Registros salvos: {len(df_filtrado)}")
        
        return df_filtrado
        
    except Exception as e:
        print(f"❌ Erro ao salvar arquivo: {e}")
        return None

def verificar_integridade_filtro(df_original, df_filtrado, outliers_ids):
    """
    Verificar a integridade do filtro aplicado
    """
    print(f"\n" + "="*60)
    print("VERIFICAÇÃO DE INTEGRIDADE")
    print("="*60)
    
    # Criar versão limpa dos IDs do DataFrame filtrado para verificação
    df_filtrado_check = df_filtrado.copy()
    df_filtrado_check['ID_IMG_CLEAN'] = df_filtrado_check['ID_IMG'].astype(str).apply(lambda x: x.split('.')[0] if '.' in x else x)
    
    # Verificar se algum outlier ainda está presente
    outliers_restantes = df_filtrado_check['ID_IMG_CLEAN'].isin(outliers_ids).sum()
    print(f"✅ Outliers restantes no arquivo filtrado: {outliers_restantes}")
    
    # Verificar estatísticas por diagnóstico (se coluna existe)
    if 'DIAG' in df_original.columns and 'DIAG' in df_filtrado.columns:
        print(f"\n📊 IMPACTO POR DIAGNÓSTICO:")
        
        diag_original = df_original['DIAG'].value_counts().sort_index()
        diag_filtrado = df_filtrado['DIAG'].value_counts().sort_index()
        
        for diag in diag_original.index:
            original_count = diag_original.get(diag, 0)
            filtrado_count = diag_filtrado.get(diag, 0)
            removidos = original_count - filtrado_count
            perc_removidos = (removidos / original_count * 100) if original_count > 0 else 0
            
            print(f"   • {diag}: {original_count} → {filtrado_count} "
                  f"(removidos: {removidos}, {perc_removidos:.1f}%)")
    
    # Verificar estatísticas por paciente (se coluna existe)
    if 'ID_PT' in df_original.columns and 'ID_PT' in df_filtrado.columns:
        print(f"\n📊 IMPACTO POR PACIENTE:")
        pacientes_original = df_original['ID_PT'].nunique()
        pacientes_filtrado = df_filtrado['ID_PT'].nunique()
        pacientes_removidos = pacientes_original - pacientes_filtrado
        
        print(f"   • Pacientes originais: {pacientes_original}")
        print(f"   • Pacientes restantes: {pacientes_filtrado}")
        print(f"   • Pacientes removidos: {pacientes_removidos}")

# Função principal
def main():
    # Definir caminhos dos arquivos
    csv_path = "output/adni/adnimerged.csv"
    txt_path = "input/adni/outliers_adni.txt"
    output_path = "output/adni/adnimerged.csv"
    
    # Executar filtro
    df_filtrado = remover_outliers_adni(csv_path, txt_path, output_path)
    
    if df_filtrado is not None:
        # Carregar outliers para verificação
        try:
            with open(txt_path, 'r') as file:
                outliers_ids = []
                for line in file:
                    line = line.strip()
                    if line and line.startswith('I'):
                        # Processar igual ao código principal
                        if '.' in line:
                            clean_id = line.split('.')[0]
                            outliers_ids.append(clean_id)
                        else:
                            outliers_ids.append(line)
            
            # Carregar DataFrame original para comparação
            df_original = pd.read_csv(csv_path)
            
            # Verificar integridade
            verificar_integridade_filtro(df_original, df_filtrado, outliers_ids)
            
        except Exception as e:
            print(f"⚠️ Não foi possível realizar verificação de integridade: {e}")
        
        print(f"\n🎉 PROCESSO CONCLUÍDO COM SUCESSO!")
        print(f"   Arquivo filtrado disponível em: {output_path}")
    else:
        print(f"\n❌ PROCESSO FALHOU")

if __name__ == "__main__":
    main()

FILTRO DE OUTLIERS - ADNI


✅ CSV carregado: 12931 registros
   Arquivo: output/adni/adnimerged.csv
✅ TXT carregado: 12 IDs de outliers
   Arquivo: input/adni/outliers_adni.txt
   Exemplos processados: ['I416112', 'I35210', 'I436253', 'I74064', 'I135611']

📊 ANÁLISE DE OUTLIERS:
   • IDs de outliers no TXT: 12
   • Outliers encontrados no CSV: 10
   • Exemplos encontrados:
     - I23662 -> I23662 (Paciente: 031_S_0821)
     - I35210 -> I35210 (Paciente: 133_S_1170)
     - I59843 -> I59843 (Paciente: 133_S_0488)

🔄 FILTRO APLICADO:
   • Registros originais: 12931
   • Registros removidos: 10
   • Registros restantes: 12921
   • Redução: 0.08%

⚠️ OUTLIERS NÃO ENCONTRADOS NO CSV: 2
   Exemplos: ['I416112', 'I416217']

✅ Arquivo filtrado salvo: output/adni/adnimerged.csv
   Registros salvos: 12921

VERIFICAÇÃO DE INTEGRIDADE
✅ Outliers restantes no arquivo filtrado: 0

📊 IMPACTO POR DIAGNÓSTICO:
   • AD: 3205 → 3205 (removidos: 0, 0.0%)
   • CN: 3966 → 3966 (removidos: 0, 0.0%)
   • MCI: 5750 → 5750 (removidos: 0, 0

## Filtros específicos

In [9]:
# RUN CRITERIA

output_path = 'output/adni/'

print(f'''
#####################################
# Critérios de inclusão de exclusão #
#####################################
      ''')

# Caminhos e parâmetros
input_criteria_path = output_path + 'adnimerged.csv'
ad_mci_time_diff = 12 # Diferença de tempo em meses entre o diagnóstico de MCI e AD;
mfs = 1.5             # 1.5 ou 3.0, Magnetic Field Strength, ie, Intensidade do campo magnético;
time_range = 36       # Intervalo de tempo em meses para a janela deslizante;
min_img = 3           # Número mínimo de imagens para a janela deslizante;
num_img = min_img     #  Número de imagens para a análise combinatória;
sex = 'both'          # 'both' ou 'F' ou 'M', ie, Sexo dos pacientes, todos, feminino ou masculino;
abordagem = 4         # 1=combinatória, 2=sequências consecutivas, 3=primeira janela+combinatória, 4=primeira janela+consecutivas

# Chamada da função main_criteria
cn_dict, smci_dict, pmci_dict = main_criteria(input_criteria_path, ad_mci_time_diff, mfs, 
                                          time_range, min_img, num_img, sex, output_path, abordagem)




#####################################
# Critérios de inclusão de exclusão #
#####################################
      
Planilha adnimerged.csv:
Imagens rotuladas: 12921
--------------------------------------------------


/tmp/ipykernel_272400/351553283.py:115: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  result_df = df.groupby(["ID_PT", "MRI_DATE"], group_keys=False).apply(filter_group)


Filtro de diagnósticos CN:

Quantidade de pacientes com diagnóstico CN: 404
Quantidade de imagens com diagnóstico CN: 1864
--------------------------------------------------
Filtro de diagnósticos sMCI e pMCI:

Subconjuntos MCI = sMCI + pMCI:
Pacientes 458 sMCI + 220 pMCI: 678
Imagens 1960 sMCI + 1363 pMCI: 3323
--------------------------------------------------
Filtro do primeiro diagnóstico AD para pacientes pMCI:

Imagens pacientes pmci (todos AD): 1363
Imagens pacientes pmci (apenas 1º AD): 989
Imagens AD removidas: 374
--------------------------------------------------
Filtro de pacientes pMCI com base no tempo entre o último MCI e o primeiro AD:

Quantidade de pacientes pMCI após filtro de tempo: 220
Quantidade de imagens pMCI após filtro de tempo: 914
--------------------------------------------------
Antes do filtro MFS (no_duplication_filter):

CN   — pacientes: 404, imagens: 1864
sMCI — pacientes: 458, imagens: 1960
pMCI — pacientes: 220, imagens: 914
------------------------

/tmp/ipykernel_272400/351553283.py:231: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_resolved = df.groupby('ID_IMG', group_keys=False).apply(resolve_conflict)


Dados de imagem salvos em output/adni/image_data.txt
Conjuntos salvos em output/adni/cj_data_abordagem_4.txt (abordagem 4)
✅ TIME_PROG calculado (meses inteiros) e salvo em output/adni/cj_data_abordagem_4.txt
